
# 레슨 03 — 실습 미션 정답지

> 🔒 **교사·관리자 전용. 학생에게 배포 금지.**

## 환경 셀

In [ ]:
import os
import pandas as pd

IS_COLAB = "COLAB_GPU" in os.environ or "COLAB_TPU_ADDR" in os.environ
DATA_BASE = "./data"
print("data base:", DATA_BASE)

---

## 미션 1 정답 — 데이터 첫 점검

In [ ]:
df = pd.read_csv(f"{DATA_BASE}/sales_orders.csv")
print("shape:", df.shape)
print("columns:", list(df.columns))
print("dtypes:\n", df.dtypes)
print(df.head(8))
print(df.tail(3))
print(df.info())
print(df.describe())

**채점 포인트**

- `df.shape` 가 `(1000, 9)` 인지 확인한다.
- `info()` 출력에서 결측치가 없고 숫자 열이 숫자형인지 확인했는지 본다.
- `describe()` 의 결과를 단순 출력만 하지 말고 어떤 숫자 열이 요약되는지 말할 수 있어야 한다.

---

## 미션 2 정답 — Series 와 DataFrame 구분

In [ ]:
qty = df["quantity"]
selected = df[["order_id", "category", "quantity", "unit_price"]]

print(type(qty))
print(qty.head())
print(type(selected))
print(selected.head())
print("수량 평균:", qty.mean())
print("수량 최댓값:", qty.max())
print("수량 최솟값:", qty.min())
print("카테고리 고유값:", df["category"].unique())

**해설**: 열 하나를 문자열로 꺼내면 Series, 열 이름 리스트로 꺼내면 DataFrame 이다. 학생이 `df["quantity"]` 와 `df[["quantity"]]` 의 타입을 비교하면 개념이 빨리 잡힌다.

---

## 미션 3 정답 — 조건 선택

In [ ]:
app_orders = df[df["channel"] == "A"]
vip_gadget = df[(df["customer_grade"] == "V") & (df["category"] == "GD")]
discount_orders = df[df["discount_rate"] >= 0.10]

print("앱 주문 수:", len(app_orders))
print(app_orders.head())
print("VIP 기기 주문 수:", len(vip_gadget))
print(vip_gadget.head())
print("10% 이상 할인 주문 수:", len(discount_orders))
print(discount_orders.head())

**채점 포인트**

- 조건이 두 개인 곳에서 괄호와 `&` 를 사용했는지 확인한다.
- `and` 를 사용하면 `ValueError: The truth value of a Series is ambiguous` 가 발생한다.
- `discount_rate > 0.10` 으로 쓴 학생은 10% 할인 주문을 제외하므로 요구사항과 다르다.

---

## 미션 4 정답 — 계산 열과 상위 주문

In [ ]:
df["gross_revenue"] = df["quantity"] * df["unit_price"]
df["net_revenue"] = df["gross_revenue"] * (1 - df["discount_rate"])

print("총 순매출:", f"{df['net_revenue'].sum():,.0f}원")
print("평균 주문금액:", f"{df['net_revenue'].mean():,.0f}원")
print("중앙 주문금액:", f"{df['net_revenue'].median():,.0f}원")

top10 = df.sort_values("net_revenue", ascending=False).head(10)
print(top10[["order_id", "category", "quantity", "unit_price", "discount_rate", "net_revenue"]])

**해설**: 할인 후 매출은 할인율을 빼서 곱해야 한다. `gross_revenue * discount_rate` 는 할인 금액이지 순매출이 아니다.

---

## 미션 5 정답 — 분포 요약과 결론

In [ ]:
channel_counts = df["channel"].value_counts()
category_counts = df["category"].value_counts()
channel_ratio = df["channel"].value_counts(normalize=True) * 100

print("채널 주문 수:\n", channel_counts)
print("카테고리 주문 수:\n", category_counts)
print("채널 비율(%):\n", channel_ratio.round(1))

best_channel = channel_counts.idxmax()
best_category = category_counts.idxmax()
print("주문 수 1위 채널:", best_channel)
print("주문 수 1위 카테고리:", best_category)

**결론 예시**

In [ ]:
%%markdown
## 결론

총 순매출은 코드 출력 기준으로 계산되며, 평균 주문금액이 중앙 주문금액보다 크다면 일부 고가 주문이 평균을 끌어올렸다고 볼 수 있다.
주문 수 기준 핵심 채널과 핵심 카테고리는 각각 `value_counts()` 의 1위 값이므로, 다음 분석에서는 이 조합을 우선 살펴보겠다.

---

## 보너스 정답

In [ ]:
region_counts = df["region"].value_counts()
print(region_counts)
print("주문 수 1위 지역:", region_counts.idxmax())

In [ ]:
no_discount = df[df["discount_rate"] == 0]
with_discount = df[df["discount_rate"] > 0]
print("무할인 평균 주문금액:", no_discount["net_revenue"].mean())
print("할인 평균 주문금액:", with_discount["net_revenue"].mean())

In [ ]:
print("loc 행 수:", len(df.loc[:5]))
print("iloc 행 수:", len(df.iloc[:5]))

`df.loc[:5]` 는 인덱스 라벨 0~5 를 포함해 6행, `df.iloc[:5]` 는 위치 0~4 까지 5행이다.

---

## 학생 답안에서 자주 보는 패턴

| 패턴 | 의미 | 교사 코멘트 |
|---|---|---|
| `df['a','b']` | 여러 열 선택 문법 혼동 | `df[['a', 'b']]` 로 고치게 함 |
| `and` 사용 | Series 조건을 스칼라 조건처럼 생각 | `&` 와 괄호를 다시 설명 |
| `net_revenue = gross * discount_rate` | 할인액과 순매출 혼동 | 10% 할인 예시로 손계산 |
| `sort_values` 후 원본이 바뀌었다고 생각 | 기본은 새 DataFrame 반환 | `inplace` 는 초반에 쓰지 말라고 안내 |
| 결론에 숫자가 없음 | 출력 복기만 함 | 총매출/1위/비율 중 하나 이상 넣게 함 |

---

## 추가 채점 메모

이번 레슨은 pandas 첫 수업이므로, 학생 답안이 모범 답안과 줄 단위로 같을 필요는 없다. 아래 기준을 우선 적용한다.

1. **구조 점검을 했는가**: `head()` 만 보고 넘어간 답안은 부족하다. `shape`, `columns`, `dtypes` 또는 `info()` 중 최소 2개 이상이 있어야 한다.
2. **타입 구분을 이해했는가**: Series 와 DataFrame 을 정확히 말하지 못해도, 열 하나와 여러 열을 서로 다른 문법으로 꺼냈으면 통과 가능하다.
3. **조건식이 안전한가**: `&` 와 괄호가 핵심이다. 우연히 결과가 나왔더라도 조건 우선순위가 틀리면 피드백한다.
4. **계산 열 공식이 맞는가**: `net_revenue` 공식 오류는 뒤 결과를 모두 왜곡하므로 반드시 수정 후 통과 처리한다.
5. **결론이 데이터에 근거하는가**: 결론 셀에 숫자나 1위 항목이 전혀 없으면 미완료로 본다.

## 수업 중 빠른 확인용 질문

- `df["quantity"]` 와 `df[["quantity"]]` 중 Series 는 어느 쪽인가?
- `.loc[:5]` 와 `.iloc[:5]` 중 6행이 나오는 것은 어느 쪽인가?
- 할인율 20% 주문의 순매출은 정가의 몇 % 인가?
- `value_counts(normalize=True)` 결과에 100을 곱하면 어떤 단위가 되는가?

학생이 이 네 질문에 답하면 03강 핵심 개념은 대체로 통과한 것이다.

## 보충 설명

`value_counts()` 는 이번 강의에서 처음 만나는 “범주형 요약” 도구다. 평균을 낼 수 없는 문자열 열도 개수를 세면 중요한 패턴이 보인다. 이후 07강의 `groupby` 는 이 생각을 더 일반화한 도구라고 예고하면 연결이 자연스럽다.